In [2]:
# 1. Setup de caminhos locais
import os
import sys
import subprocess
from datetime import datetime
from pathlib import Path
import json

# Adiciona src ao path (notebook está em notebooks/)
sys.path.insert(0, str(Path(".").absolute().parent))

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "13_etl_dedup"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)

SNAPSHOT_ID = STAMP

BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [3]:
# 2. Carregar dados de entrada (fonte única oficial TCC)
import pandas as pd

# FONTE OFICIAL TCC: news_multisource.parquet gerado no notebook 12
f_parquet = os.path.join(RAW_PATH, "news_multisource.parquet")
if not os.path.exists(f_parquet):
    raise FileNotFoundError(
        f"Base oficial não encontrada: {f_parquet}\n"
        "Execute o notebook 12_data_collection_multisource.ipynb em modo FULL primeiro."
    )

df_raw = pd.read_parquet(f_parquet)
print("="*80)
print("FONTE OFICIAL TCC CARREGADA")
print("="*80)
print(f"Arquivo: {f_parquet}")
print(f"Total de registros: {len(df_raw):,}")
print(f"Dias distintos: {df_raw['date'].nunique():,}")
print(f"Período: {df_raw['date'].min()} → {df_raw['date'].max()}")
print(f"Fontes: {df_raw['source'].nunique():,}")
print("="*80)
display(df_raw.head(3))

FONTE OFICIAL TCC CARREGADA
Arquivo: C:\Users\ander\OneDrive\TCC_USP\data_raw\news_multisource.parquet
Total de registros: 91,959
Dias distintos: 2,771
Período: 2018-01-02 00:00:00 → 2025-11-19 00:00:00
Fontes: 508


,date,source,title,url,text_full
0,2018-01-02,g1.globo.com,Bovespa começa 2018 no azul apoiada em ações d...,https://g1.globo.com/economia/noticia/bovespa-...,Bovespa começa 2018 no azul apoiada em ações d...
1,2018-01-02,jj.com.br,"Com alta de 504 % em 2017 , Magazine Luiza ent...",http://www.jj.com.br/noticias-52569-com-alta-d...,"Com alta de 504 % em 2017 , Magazine Luiza ent..."
2,2018-01-02,tribunahoje.com,Ibovespa estica rali e abre 2018 com pontuação...,http://tribunahoje.com/noticias/economia/2018/...,Ibovespa estica rali e abre 2018 com pontuação...


In [4]:
# 3. Normalizar esquema (COALESCE robusto: sem .str em DataFrame)
import pandas as pd, numpy as np, re

COLS = ["date","title","text","source","url","origin"]

def get_series(df, name):
    """Retorna uma Series mesmo se houver colunas duplicadas com o mesmo nome."""
    mask = (df.columns == name)
    if mask.sum() == 0:
        return pd.Series([None]*len(df), index=df.index, dtype='object')
    sub = df.loc[:, mask]
    if sub.shape[1] == 1:
        return sub.iloc[:, 0]
    # Coalesce por linha: 1º valor não-vazio/não-NaN
    return sub.apply(lambda row: next((x for x in row if pd.notna(x) and str(x).strip() != ""), None), axis=1)

def coalesce_synonyms(df, candidates):
    """Coalesce de vários nomes sinônimos -> 1 Series 'limpa'."""
    s = pd.Series([None]*len(df), index=df.index, dtype='object')
    for name in candidates:
        col = get_series(df, name)   # sempre Series
        mask = s.isna() | (s.astype(str).str.strip().isin(['', 'nan', 'None']))
        s = s.where(~mask, col)
    return s

# Sinônimos por coluna
date_candidates   = ["date","data","dt","pubDate","published","published_at","published_at_utc","publishedAt","Date"]
title_candidates  = ["title","Title","titulo","headline"]
text_candidates   = ["text","descricao","resumo","summary"]
source_candidates = ["source","SourceCommonName","source_name","fonte"]
url_candidates    = ["url","link","DocumentIdentifier"]
origin_candidates = ["origin"]

# Coalesce de cada campo
s_date   = coalesce_synonyms(df_raw, date_candidates)
s_title  = coalesce_synonyms(df_raw, title_candidates)
s_text   = coalesce_synonyms(df_raw, text_candidates)
s_source = coalesce_synonyms(df_raw, source_candidates)
s_url    = coalesce_synonyms(df_raw, url_candidates)
s_origin = coalesce_synonyms(df_raw, origin_candidates)

# Parse de datas (ISO, GDELT yyyymmddHHMMSS, fallback dd/MM/yyyy)
str_date = s_date.astype(str)

mask14 = str_date.str.fullmatch(r"\d{14}", na=False)              # ex.: 20250131094530
d1 = pd.to_datetime(str_date.where(mask14), format="%Y%m%d%H%M%S", errors="coerce")
d2 = pd.to_datetime(str_date.where(~mask14), errors="coerce")
date_parsed = d1.combine_first(d2)

if date_parsed.notna().sum() == 0 and str_date.str.contains(r"/").any():
    date_parsed = pd.to_datetime(str_date, format="%d/%m/%Y", errors="coerce")

# Monta dataframe final (Series -> strings limpas)
df = pd.DataFrame({
    "date":   date_parsed,
    "title":  s_title.astype(str).fillna("").str.strip(),
    "text":   s_text.astype(str).fillna("").str.strip(),
    "source": s_source.astype(str).fillna("").str.strip(),
    "url":    s_url.astype(str).fillna("").str.strip(),
    "origin": s_origin.astype(str).fillna("").str.strip()
})

# Remove inválidos e mostra amostra
before = df.shape[0]
df = df.dropna(subset=["date"]).copy()
df = df[df["title"].str.len() > 0].copy()
print(f"Removidas {before - df.shape[0]} linhas inválidas (sem data/sem título).")
print("Colunas finais:", df.columns.tolist())
display(df.head(3))

Removidas 0 linhas inválidas (sem data/sem título).
Colunas finais: ['date', 'title', 'text', 'source', 'url', 'origin']


,date,title,text,source,url,origin
0,2018-01-02,Bovespa começa 2018 no azul apoiada em ações d...,None,g1.globo.com,https://g1.globo.com/economia/noticia/bovespa-...,None
1,2018-01-02,"Com alta de 504 % em 2017 , Magazine Luiza ent...",None,jj.com.br,http://www.jj.com.br/noticias-52569-com-alta-d...,None
2,2018-01-02,Ibovespa estica rali e abre 2018 com pontuação...,None,tribunahoje.com,http://tribunahoje.com/noticias/economia/2018/...,None


In [5]:
# 4. Normalizar URL (remover UTM/trailing e padronizar host)
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode

def normalize_url(u: str) -> str:
    if not isinstance(u, str) or len(u.strip()) == 0 or u.strip().lower() in ["nan", "none"]:
        return ""
    try:
        p = urlparse(u.strip())
        # normalizar host
        netloc = p.netloc.lower()
        # filtrar query (remover utm_*, fbclid, gclid, etc.)
        keep_q = []
        for k,v in parse_qsl(p.query, keep_blank_values=False):
            if not (k.lower().startswith("utm_") or k.lower() in {"fbclid","gclid","mc_cid","mc_eid"}):
                keep_q.append((k,v))
        q = urlencode(keep_q, doseq=True)
        # remover barra final redundante
        path = p.path.rstrip("/")
        norm = urlunparse((p.scheme.lower(), netloc, path, "", q, ""))
        return norm
    except Exception:
        return u.strip()

df["url_norm"] = df["url"].apply(normalize_url)

In [6]:
# 5. Deduplicação (chave URL; fallback data+título)
has_url = df["url_norm"].str.len() > 0
df["uid"] = np.where(
    has_url,
    df["url_norm"],
    df["date"].dt.strftime("%Y-%m-%d") + "_" + df["title"].str[:80].str.replace(r"\W+", "_", regex=True)
)

before = df.shape[0]
df = df.drop_duplicates(subset=["uid"]).reset_index(drop=True)
removed = before - df.shape[0]
print(f"Duplicatas removidas: {removed}. Linhas finais: {df.shape[0]}")

Duplicatas removidas: 18. Linhas finais: 91941


In [7]:
# 6. Qualidade (missing, origem, datas)
print("Faltantes por coluna (%):")
display((df[["title","text","source","url_norm","origin"]].isna().mean()*100).round(2))

print("Registros por origem:")
display(df["origin"].fillna("desconhecida").value_counts())

print("Faixa de datas:")
print(df["date"].min(), "→", df["date"].max())

Faltantes por coluna (%):


title       0.0
text        0.0
source      0.0
url_norm    0.0
origin      0.0
dtype: float64

Registros por origem:


origin
None    91941
Name: count, dtype: int64

Faixa de datas:
2018-01-02 00:00:00 → 2025-11-19 00:00:00


In [8]:
# 7. Salvar limpo + dicionário de dados + relatório + VALIDAÇÃO TCC
clean_parquet = os.path.join(INTERIM_PATH, "news_clean_multisource.parquet")
clean_csv     = os.path.join(INTERIM_PATH, "news_clean_multisource.csv")
dict_csv      = os.path.join(PROC_PATH, "dicionario_dados_news.csv")
report_json   = os.path.join(PROC_PATH, f"etl_report_{SNAPSHOT_ID}.json")

# salvar dados limpos
df_out = df[["date","title","text","source","url_norm","origin"]].rename(columns={"url_norm":"url"})

# ========== VALIDAÇÃO OBRIGATÓRIA TCC ==========
MIN_DAYS_THRESHOLD = 200
n_days_final = df_out["date"].nunique()
n_records_final = len(df_out)
date_min = df_out["date"].min()
date_max = df_out["date"].max()

print("\n" + "="*80)
print("VALIDAÇÃO DA BASE LIMPA (TCC)")
print("="*80)
print(f"Total de registros: {n_records_final:,}")
print(f"Dias distintos: {n_days_final:,}")
print(f"Período: {date_min} → {date_max}")
print(f"Fontes: {df_out['source'].nunique():,}")

if n_days_final < MIN_DAYS_THRESHOLD:
    raise RuntimeError(
        f"❌ BASE INSUFICIENTE PARA TCC!\n"
        f"   Dias encontrados: {n_days_final}\n"
        f"   Mínimo exigido: {MIN_DAYS_THRESHOLD}\n"
        f"   Execute o notebook 12 em modo FULL para coletar base histórica completa."
    )

print(f"\n✅ VALIDAÇÃO APROVADA: {n_days_final:,} dias >= {MIN_DAYS_THRESHOLD}")
print("   STATUS: Base adequada para TCC")
print("="*80 + "\n")
# ================================================

df_out.to_parquet(clean_parquet, index=False)
df_out.to_csv(clean_csv, index=False, encoding="utf-8")

# dicionário de dados
dic = pd.DataFrame([
    {"variavel":"date","tipo":"datetime64[ns]","descricao":"Data/hora de publicação normalizada UTC"},
    {"variavel":"title","tipo":"string","descricao":"Título/manchete da notícia"},
    {"variavel":"text","tipo":"string","descricao":"Resumo/descrição curta (se disponível)"},
    {"variavel":"source","tipo":"string","descricao":"Nome da fonte (domínio normalizado)"},
    {"variavel":"url","tipo":"string","descricao":"URL normalizada (sem parâmetros de tracking)"},
    {"variavel":"origin","tipo":"string","descricao":"Origem técnica de coleta (gdelt, newsapi, etc.)"},
], columns=["variavel","tipo","descricao"])
dic.to_csv(dict_csv, index=False, encoding="utf-8")

# relatório
report = {
    "notebook": NB_NAME,
    "snapshot_id": SNAPSHOT_ID,
    "inputs": {
        "raw_rows": int(df_raw.shape[0]),
        "raw_sources": ["news_multisource.parquet"]
    },
    "output_rows": int(df_out.shape[0]),
    "output_days": int(n_days_final),
    "date_range": f"{date_min} → {date_max}",
    "duplicates_removed": int(removed),
    "validation_passed": n_days_final >= MIN_DAYS_THRESHOLD,
    "paths": {
        "clean_parquet": clean_parquet,
        "clean_csv": clean_csv,
        "data_dictionary": dict_csv
    }
}
with open(report_json, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("✅ Arquivos salvos:")
print(f"   {clean_parquet}")
print(f"   {clean_csv}")
print(f"   {dict_csv}")
print(f"   {report_json}")


VALIDAÇÃO DA BASE LIMPA (TCC)
Total de registros: 91,941
Dias distintos: 2,771
Período: 2018-01-02 00:00:00 → 2025-11-19 00:00:00
Fontes: 508

✅ VALIDAÇÃO APROVADA: 2,771 dias >= 200
   STATUS: Base adequada para TCC

✅ Arquivos salvos:
   C:\Users\ander\OneDrive\TCC_USP\data_interim\news_clean_multisource.parquet
   C:\Users\ander\OneDrive\TCC_USP\data_interim\news_clean_multisource.csv
   C:\Users\ander\OneDrive\TCC_USP\data_processed\dicionario_dados_news.csv
   C:\Users\ander\OneDrive\TCC_USP\data_processed\etl_report_20251119_215553.json
✅ Arquivos salvos:
   C:\Users\ander\OneDrive\TCC_USP\data_interim\news_clean_multisource.parquet
   C:\Users\ander\OneDrive\TCC_USP\data_interim\news_clean_multisource.csv
   C:\Users\ander\OneDrive\TCC_USP\data_processed\dicionario_dados_news.csv
   C:\Users\ander\OneDrive\TCC_USP\data_processed\etl_report_20251119_215553.json


In [30]:
# 8. Resumo final (para colar no README de resultados)
from IPython.display import display, Markdown

rows_by_origin = df_out["origin"].fillna("desconhecida").value_counts().to_dict()
md = f"""
**{NB_NAME} – ETL & Dedup concluído**

- Linhas limpas: **{df_out.shape[0]}**
- Duplicatas removidas: **{removed}**
- Período: **{df_out['date'].min().date()} → {df_out['date'].max().date()}**
- Por origem: **{rows_by_origin}**

Arquivos:
- `data_processed/news_multisource_clean.parquet`
- `data_processed/news_multisource_clean.csv`
- `data_processed/dicionario_dados_news.csv`
- `data_processed/etl_report_{SNAPSHOT_ID}.json`
"""
display(Markdown(md))
print("Feito ✅")


**13_etl_dedup – ETL & Dedup concluído**

- Linhas limpas: **100**
- Duplicatas removidas: **0**
- Período: **2025-09-19 → 2025-10-17**
- Por origem: **{'google_rss': 100}**

Arquivos:
- `data_processed/news_multisource_clean.parquet`
- `data_processed/news_multisource_clean.csv`
- `data_processed/dicionario_dados_news.csv`
- `data_processed/etl_report_20251018_152153.json`


Feito ✅
